# Operators and Expressions

**Topic:** `1.6` &nbsp;|&nbsp; **Module 1: Getting Started with Python**

*The small set of symbols Python puts between values, what each one returns, how precedence groups them, and how a packed status word is read and written with bitwise operators.*

> *Course:* Python Master Course &nbsp;|&nbsp; **Project:** ROBO-X (M1)

### Navigate

* [Lesson](lesson.ipynb) &nbsp;|&nbsp; [Exercises](exercises.ipynb) &nbsp;|&nbsp; [Solutions](solution.ipynb) &nbsp;|&nbsp; [Research](research.ipynb) &nbsp;|&nbsp; [Quiz](quiz.ipynb) &nbsp;|&nbsp; [Mini-project](mini_project.ipynb) &nbsp;|&nbsp; [Robotics challenge](robotics_challenge.ipynb)

Module index: [01_getting_started/README.md](../README.md)

## 1. Why This Topic Matters

Operators are the vocabulary of every expression you will write, and two of them - `and` and `or` - do not behave the way their names suggest. On top of that, firmware communicates state as a single packed integer, so the bitwise family is a practical requirement for anyone working with a real robot.

## 2. Learning Objectives

By the end of this topic you will be able to:

1. Predict the return type of every operator family.
2. Explain how precedence groups a mixed expression, and parenthesise deliberately.
3. Use `and` and `or` as value-producing expressions, not just booleans.
4. Apply bitwise masks to set, clear, toggle and test individual flags.
5. Build a defined-bit mask so unknown flags are reported rather than lost.
6. Interpret an expression safely with ast instead of eval.

## 3. Prerequisites

This topic assumes you already have:

- Topic 1.5 Core Data Types and Type Conversion

## 4. Mental Model

**Mental model: an expression is a small machine with one output.** Each operator is a stage. Precedence decides the order the stages run, and the type of the value leaving each stage decides what the next stage is allowed to accept. When an expression gives the wrong answer, you are looking for a stage running in the wrong order, or a stage fed the wrong kind of value.

| You write | Runs as | Easy to miss |
| --- | --- | --- |
| `a + b == c` | `(a + b) == c` | comparison binds loosest |
| `not a == b` | `not (a == b)` | `not` is looser than `==` |
| `x and y` | returns `x` or `y` | an operand, not a bool |
| `x or y` | returns `x` or `y` | short-circuits |
| `a & b == 0` | `(a & b) == 0` | bitwise binds tighter than `==` |
| `a if cond else b` | one of the two values | an expression, not a statement |

## 5. Terminology

Vocabulary used consistently throughout the course.

| Term | Meaning |
| --- | --- |
| Operator | A symbol performing an operation, such as `+` or `==`. |
| Expression | Code that produces a value. |
| Statement | Code that performs an action; does not necessarily produce a value. |
| Precedence | The rules that decide how operators group. |
| Short-circuit | Stopping evaluation once the result is known. |
| Floor division | `//`, which rounds towards minus infinity. |
| Modulo | `%`, the remainder after floor division. |
| Chained comparison | `0 <= x <= 100`, evaluated as a single chained call. |
| Bitmask | An integer whose set bits mark positions of interest. |
| Walrus operator | `:=`, which binds a name inside an expression. |
| Operator overloading | Defining dunder methods so your class supports an operator. |

## 6. Conceptual Explanation

Python's operators fall into a small number of families, and each family has a different return type. Arithmetic operators return numbers. Comparison operators return a bool. The membership tests return a bool. The bitwise operators work on integers bit by bit. The logical operators `and`, `or` and `not` are the odd ones out: they return one of their *operands*, not necessarily a bool. That single fact explains a great deal of otherwise mysterious behaviour.

In [ ]:
a, b = 48, 16
print('arithmetic :', a + b, a - b, a * b, a / b, a // b, a % b, a ** 2)
print('division   : 48 / 16 =', 48 / 16, type(48 / 16).__name__)
print('comparison :', a > b, a == b, a != b)
print('membership :', 16 in (16, 32), 64 in (16, 32))
print('bitwise    :', a & b, a | b, a ^ b, a << 1, a >> 1)

`and` and `or` short-circuit and return an operand. That is a feature, not a quirk: it is how Python expresses a default value without a ternary, and how a guard clause protects an expression from a division that would fail.

In [ ]:
label = ''
print('or returns  :', repr(label or 'unnamed'))   # returns the right operand
name = 'rover-01'
print('and returns :', repr(name and name.upper()))

print('bool() them :', bool(label or 'unnamed'), bool(name and name.upper()))

total = 0
print('short circuit:', total or 1 / 0 if False else (total or 'default'))
print('division safe:', (total or 1) and 'computed')

> **The walrus operator**
>
> `:=` binds a name as part of an expression, so `if (n := len(items)) > 10:` computes a length once and tests it. Use it when it removes a duplicated computation, not because it is shorter.

## 7. Formal Theory

Precedence determines grouping. In Python it follows one rule: **everything else binds tighter than comparison**. So `1 + 2 == 3` is `(1 + 2) == 3`, and `not a == b` is `not (a == b)`, not `(not a) == b`. Bitwise `&`, `|` and `^` bind *tighter* than comparison but *looser* than arithmetic - the reverse of C, which surprises programmers arriving from other languages.

```text
lambda < or < and < not < in/is/<=/>/!=/== < | < ^ < & < <<,>> < +,- < *,/,//,% < ** < unary < ()
```

Operators are not built into the language. Each one is a method call: `a + b` invokes `a.__add__(b)`, and `a < b` invokes `a.__lt__(b)`. That is why a class you write can support `+` by defining `__add__`, and it is also why operator overloading can surprise a reader who assumes `+` always means addition.

In [ ]:
import dis


def add_then_compare(a, b):
    return a + b == 3


print('source :', 'a + b == 3')
print('values :', add_then_compare(1, 2))
print('co_names:', add_then_compare.__code__.co_names)

for instruction in dis.get_instructions(add_then_compare):
    if instruction.opname in {'BINARY_OP', 'COMPARE_OP'}:
        print('  ', instruction.opname, instruction.argval)

| Family | Operators | Returns |
| --- | --- | --- |
| Arithmetic | `+ - * / // % **` | a number |
| Comparison | `== != < <= > >=` | a bool |
| Chained | `0 <= x <= 100` | a bool |
| Logical | `and or not` | **an operand** |
| Membership | `in`, `not in` | a bool |
| Bitwise | `& \| ^ ~ << >>` | an int |
| Identity | `is`, `is not` | a bool |

## 8. Syntax

An **expression** produces a value; a **statement** does something with it. Almost every operator is part of an expression, and most expressions can be used as a statement - but a statement is not an expression, which is why you cannot write `x = if a else b`.

In [ ]:
battery_wh = 48.0
max_speed_mps = 1.5

# comparison chains read as mathematics does
in_range = 0 < battery_wh <= 100
print('chain       :', in_range)

# the conditional expression, used as a value
verdict = 'LOW' if battery_wh < 20 else 'OK'
print('conditional :', verdict)

# the walrus operator binds a name inside the expression
if (n := len('rover-01')) > 3:
    print('walrus      :', n)

The anti-pattern:

```text
if not battery_wh > 20:      # double negative: hard to read
    alert()

if battery_wh <= 20:         # say what you mean
    alert()

a & b == 0                   # binds as (a & b) == 0, NOT a & (b == 0)
a & (b == 0)                 # what C programmers often expect
```

> **Not is a keyword, not an operator**
>
> `not x` binds more loosely than comparison, so `not a == b` means `not (a == b)`. If you want the other grouping you must parenthesise it.

## 9. Basic Examples

**Example 1 - the arithmetic family in context.**

In [ ]:
distance_m = 30.0
speed_mps = 1.5
elapsed_s = 20.0

print('time needed  :', distance_m / speed_mps)
print('whole seconds:', distance_m // speed_mps)
print('remainder    :', distance_m % speed_mps)
print('scale x2     :', distance_m * 2, speed_mps ** 2)
print('unary minus  :', -distance_m, +distance_m)

**Example 2 - string operators.**

In [ ]:
name = 'rover-01'
print('concatenate :', name + '-ready')
print('repeat      :', 'ab' * 3)
print('membership  :', 'rover' in name, 'truck' in name)
print('slice       :', name[0:5], name[-2:])
print('reverse     :', name[::-1])

## 10. Intermediate Examples

Two things that surprise newcomers: division always returns a float, and a comparison chain is a single chained call that short-circuits, so `0 < x < 1` cannot blow up in the middle.

In [ ]:
print('7 / 2  ->', 7 / 2, type(7 / 2).__name__)
print('7 // 2 ->', 7 // 2, type(7 // 2).__name__)
print('7 % 2  ->', 7 % 2)

battery_pct = 50
print('0 < 50 <= 100 :', 0 < battery_pct <= 100)
print('0 < 500 <= 100:', 0 < 500 <= 100)

# negative floor division rounds towards minus infinity, not to zero
print('-7 // 2 :', -7 // 2)
print('-7 % 2  :', -7 % 2)

## 11. Advanced Examples

Bitwise operators let a compact integer carry several status flags at once, which is how firmware status words are normally stored. This is exactly the shape of a robotics status register.

In [ ]:
CHARGING = 0b0001
LOW_BATTERY = 0b0010
OBSTACLE = 0b0100
ESTOP = 0b1000

status = CHARGING | LOW_BATTERY
print('status      :', bin(status))
print('charging?   :', bool(status & CHARGING))
print('obstacle?   :', bool(status & OBSTACLE))
print('set a flag  :', bin(status | OBSTACLE))
print('clear a flag:', bin(status & ~OBSTACLE))
print('toggle      :', bin(status ^ OBSTACLE))

The three operations to remember: `| value` sets bits, `& ~value` clears them, and `^ value` toggles them. A mask is simply an integer whose set bits mark the positions of interest.

## 12. Code Walkthrough

Below is a status-word decoder. It is the standard pattern for turning a packed firmware register into a dictionary a supervisor can reason about.

In [ ]:
"""status_flags.py - decode a packed status word into named flags."""
from __future__ import annotations

CHARGING = 0b0001
LOW_BATTERY = 0b0010
OBSTACLE = 0b0100
ESTOP = 0b1000

KNOWN = {
    'CHARGING': CHARGING,
    'LOW_BATTERY': LOW_BATTERY,
    'OBSTACLE': OBSTACLE,
    'ESTOP': ESTOP,
}


def decode_status(word: int) -> dict:
    """Split a status word into active flags and unused bits.

    Returns a dict with the active flag names, the raw word, and
    any bits that are set but not defined in KNOWN.
    """
    if not isinstance(word, int) or isinstance(word, bool):
        raise TypeError('status word must be an int')
    if word < 0:
        raise ValueError('status word must not be negative')
    active = [name for name, mask in KNOWN.items() if word & mask]
    defined = 0
    for mask in KNOWN.values():
        defined |= mask
    unknown = word & ~defined
    return {
        'word': word,
        'flags': active,
        'unknown_bits': bin(unknown),
        'estop': bool(word & ESTOP),
    }

Building the `defined` mask by OR-ing every known flag is what lets the decoder report *undefined* bits. Without it, a firmware word from a newer revision would decode silently and the extra bits would vanish - one of the most expensive kinds of interoperability bug.

## 13. Line-by-Line Explanation

1. The flag constants are module-level integers written in binary so the bit layout is visible to a reader.
2. `KNOWN` maps names to masks, so decoding and encoding can share one table and the names appear in a consistent order.
3. The isinstance guard rejects bool, which would otherwise pass as an int and silently decode to no flags.
4. The negative check happens before any bit arithmetic, because `&` with a negative number produces a surprising result rather than an error.
5. The active list comprehension applies a mask per flag, which is the standard test-and-set idiom.
6. `defined` is built by OR-ing every known mask so undefined bits can be detected rather than discarded.
7. `~defined` flips all the bits, so `word & ~defined` isolates exactly the bits nobody has a name for.

## 14. Common Mistakes

**Mistake 1 - assuming bitwise and comparison have the same precedence.**

```text
flag & mask == 0        # parsed as (flag & mask) == 0
flag & (mask == 0)      # a completely different expression

flag & mask == 0        # usually what you meant; parenthesise anyway
```

**Mistake 2 - using `~` without masking.**

```text
value & ~OBSTACLE       # correct: clears the bit
value ^ OBSTACLE        # correct: toggles the bit

value & ~value          # always 0 - a common but pointless idiom
```

**Mistake 3 - double negatives.**

```text
if not battery_pct > 20:     # hard to read and easy to get wrong
    alert()

if battery_pct <= 20:        # states the rule directly
    alert()
```

**Mistake 4 - forgetting that `/` always returns a float.**

```text
7 / 2    # 3.5  (float)
7 // 2   # 3    (int)

int(7 / 2)  # 3 - convert deliberately when you mean truncation
```

## 15. Debugging Techniques

When an expression gives the wrong answer, the fault is nearly always precedence. Print the value of each sub-expression separately, or parenthesise explicitly and see whether the answer changes.

In [ ]:
flag, mask = 0b0110, 0b0010
print('flag          :', bin(flag))
print('flag & mask   :', bin(flag & mask))
print('mask == 0     :', mask == 0)
print('precedence    :', flag & mask == 0)
print('parenthesised :', (flag & mask) == 0)
print('other grouping:', flag & (mask == 0))

In [ ]:
import ast

source = 'not a == b'
tree = ast.parse(source, mode='eval')
print('source  :', source)
print('structure:', ast.dump(tree.body, indent=2)[:200])

> **The AST shows the grouping**
>
> ast.parse with mode='eval' builds the tree without evaluating anything. Printing the node types tells you exactly how the parser grouped the expression, which settles any precedence argument immediately.

## 16. Best Practices

- Parenthesise mixed-precedence expressions; do not rely on the reader's memory.
- Use `//` and `%` deliberately rather than casting the result of `/`.
- Prefer a comparison chain to `a >= b and b <= c` when it reads the same.
- Never use a bare `except` around an expression that can divide by zero.
- Name every bit constant and build masks from a single table.
- Mask with `& ~flag` when clearing; avoid the `& ~value` idiom.
- Keep the conditional expression in an assignment, not inside a call.
- Use the walrus operator only when it removes a genuine duplicate computation.

## 17. Performance Considerations

Operators are not where optimisation opportunities usually live. A bitwise test on an int is constant time; a division costs a little more than an addition but is still nanoseconds. The expression that costs you is almost always the one that calls a function or builds a collection inside a loop.

Two genuine micro-patterns are worth knowing. Replacing repeated `len(x) > 0` with a direct truth test saves a function call, and hoisting an invariant sub-expression out of a loop avoids recomputing it. Neither matters until the loop runs millions of iterations. In a robotics control loop the cost that dominates is usually I/O or the sleep at the end of the cycle, not the arithmetic, so measure before touching the expression at all.

> **Prefer clarity, then measure**
>
> Write the clearest correct expression, profile the surrounding code, and only then consider rewriting the arithmetic.

## 18. Pythonic Approaches

Accumulating with a loop versus the builtins that say what you mean.

In [ ]:
readings = [0.4, 0.6, 0.5, 0.9]

# before: an explicit accumulator
total = 0.0
for value in readings:
    total += value

# after: the intent, named
print('total :', sum(readings))
print('any   :', any(v > 0.8 for v in readings))
print('all   :', all(0 <= v <= 1 for v in readings))

# a default without a ternary
label = ''
print('label :', label or 'unnamed')

## 19. Robotics Connection

Firmware reports state as a packed word precisely so that one integer carries many independent facts. The bitwise operators are how a Python supervisor reads and writes that word, and how it must do so without disturbing the flags it does not own.

In [ ]:
import sys
from pathlib import Path

here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / 'shared').is_dir()), None)
if root is not None:
    sys.path.insert(0, str(root))

from shared.robo_x_sim import SimulatedRobot  # noqa: E402

CHARGING, LOW_BATTERY, OBSTACLE, ESTOP = 1, 2, 4, 8

robot = SimulatedRobot(name='rover-01', battery_wh=48.0)
state = robot.status()

word = 0
if state['battery_pct'] < 20:
    word |= LOW_BATTERY
if robot.read_sensor('distance') < 1.0:
    word |= OBSTACLE

print('status word :', bin(word))
print('low battery :', bool(word & LOW_BATTERY))
print('obstacle    :', bool(word & OBSTACLE))

## 20. Engineering Example

A guarded division that never raises, written so the intent is obvious at the call site.

In [ ]:
def safe_divide(numerator, denominator, default=0.0):
    """Divide, returning `default` instead of raising on a zero divisor."""
    if denominator == 0:
        return default
    return numerator / denominator


def mean_or_zero(values):
    return safe_divide(sum(values), len(values))


print('mean of three :', mean_or_zero([1.0, 2.0, 3.0]))
print('mean of empty :', mean_or_zero([]))

## 21. Guided Practice

1. Predict the type of `7 / 2`, `7 // 2` and `7 % 2` before running them.
2. Write a chained comparison that is True only for values in 0 to 100, and verify it against several inputs.
3. Build a status word with two flags set, print it in binary, then clear one flag and print it again.
4. Write an expression using `or` that supplies a default name without an `if` statement.

In [ ]:
print(7 / 2, 7 // 2, 7 % 2)

for value in (-1, 0, 50, 100, 101):
    print(f'{value:>4} in range:', 0 <= value <= 100)

word = 0b0110
print('word       :', bin(word))
print('cleared    :', bin(word & ~0b0010))

name = ''
print('default    :', name or 'unnamed')

## 22. Knowledge Checks

Answer these before moving on. Full answers are in [solution.ipynb](solution.ipynb).

- **Q1.** What does the expression `'' or 'unnamed'` evaluate to?
- **Q1.** Why build a mask of all defined bits before decoding a status word?
- **Q1.** Why should a failed distance-sensor read set the OBSTACLE flag?

## 23. Summary

Operators group into families that agree on their return type. Arithmetic returns numbers, comparison and membership return bools, bitwise operators return ints - and `and`, `or` and `not` return *operands*. That last fact is not a quirk: it is what makes `value or default` the shortest correct way to express a default, and what makes `total or 1` a safe guard before a division.

Precedence in Python has one memorable rule: everything else binds tighter than comparison. That is why `1 + 2 == 3` is `(1 + 2) == 3` and `not a == b` is `not (a == b)`. Bitwise operators sit between arithmetic and comparison, the reverse of C. Parenthesising mixed expressions costs nothing and removes a whole class of reading errors, so treat it as a habit rather than an admission that you forgot the table.

For robotics, the bitwise family deserves particular attention. Firmware packs many independent facts into one integer, and reading or writing one flag without disturbing the others is a masking exercise: `value | FLAG` sets, `value & ~FLAG` clears, `value ^ FLAG` toggles, and `value & FLAG` tests. The discipline that makes this safe is building the defined-bit mask from a single table, so an unknown bit from a newer firmware revision is reported rather than silently discarded.

## 24. Key Takeaways

- `/` always returns a float; use `//` and `%` when you mean whole numbers.
- `and` and `or` return an operand, not necessarily a bool.
- Comparison binds looser than arithmetic and than the bitwise operators.
- `not` binds looser than comparison, so `not a == b` means `not (a == b)`.
- Chained comparisons read like mathematics and short-circuit safely.
- Bitwise: `|` sets, `& ~` clears, `^` toggles, `&` tests.
- Parenthesise mixed-precedence expressions for the reader.
- Build the defined-bit mask from one table so unknown bits are reported.

## 25. Further Exploration

- [Expressions and simple statements](https://docs.python.org/3/reference/expressions.html)
- [Operator precedence table](https://docs.python.org/3/reference/expressions.html#operator-precedence)
- [Truth value testing](https://docs.python.org/3/library/stdtypes.html#truth-value-testing)
- [PEP 572 - Assignment expressions](https://peps.python.org/pep-0572/) - the walrus operator.
- [Python in a Nutshell: operators](https://github.com/dabeaz/1706dab4700b25ed0b73544bd54334b8c4c8d2b1)